# Dokumentacja Bazy Danych: nycflights13

> [!info] Kontekst
> Baza danych oparta na strukturze przypominającej schemat gwiazdy(jedna główna tabela). Główną tabelą faktów jest `flights`, a pozostałe to tabele wymiarów (słownikowe), dostarczające szczegółowego kontekstu.

## 1. Słownik Tabel

### `flights` (Tabela Główna / Tabela Faktów)
Centralny punkt projektu. Każdy wiersz reprezentuje jeden wylot z lotnisk w Nowym Jorku.
| Kolumna | Opis |
| :--- | :--- |
| **year, month, day** | Dokładna data wylotu. |
| **dep_time, arr_time** | Rzeczywisty czas wylotu i przylotu. |
| **sched_dep_time, sched_arr_time** | Planowany harmonogram lotu. |
| **dep_delay, arr_delay** | Opóźnienia w minutach (dodatnie = spóźnienie, ujemne = przed czasem). |
| **carrier** | Kod linii lotniczej (Klucz Obcy). |
| **flight** | Numer lotu. |
| **tailnum** | Numer rejestracyjny fizycznego samolotu (Klucz Obcy). |
| **origin, dest** | Kody lotniska startowego i docelowego (Klucze Obce). |
| **air_time** | Czas spędzony w powietrzu (w minutach). |
| **distance** | Pokonana odległość (w milach). |
| **hour, minute** | Godzina i minuta planowanego wylotu rozbite na osobne kolumny. |
| **time_hour** | Znacznik czasu jako jeden obiekt (przydatny do łączenia z pogodą). |

### `airlines` (Linie lotnicze)
Słownik pozwalający odszyfrować kody przewoźników.
| Kolumna | Opis |
| :--- | :--- |
| **carrier** | Dwuliterowy skrót przewoźnika (Klucz Główny). |
| **name** | Pełna, oficjalna nazwa firmy. |

### `airports` (Lotniska)
Szczegóły geograficzne i strefowe dotyczące lotnisk.
| Kolumna | Opis |
| :--- | :--- |
| **faa** | Trzyliterowy kod lotniska (Klucz Główny). |
| **name** | Pełna nazwa obiektu. |
| **lat, lon** | Współrzędne geograficzne (niezbędne do map). |
| **alt** | Wysokość n.p.m. |
| **tz, dst, tzone** | Strefa czasowa i przesunięcia czasu letniego. |

### `planes` (Samoloty)
Dane techniczne maszyn obsługujących loty.
| Kolumna | Opis |
| :--- | :--- |
| **tailnum** | Rejestracja namalowana na ogonie (Klucz Główny). |
| **year** | Rok produkcji samolotu. |
| **type** | Rodzaj budowy maszyny. |
| **manufacturer, model** | Producent (np. Boeing) i konkretny wariant modelu. |
| **engines, seats** | Ilość silników oraz miejsc na pokładzie. |
| **speed, engine** | Prędkość i typ napędu. |

### `weather` (Pogoda)
Zrzuty warunków meteo dla trzech głównych lotnisk (JFK, LGA, EWR).
| Kolumna | Opis |
| :--- | :--- |
| **origin** | Z jakiego lotniska zebrano pomiar (Część Klucza Głównego). |
| **year, month, day, hour** | Dokładny czas odczytu (Część Klucza Głównego). |
| **temp, dewp** | Temperatura i punkt rosy. |
| **humid, precip** | Wilgotność i opady. |
| **wind_dir, wind_speed, wind_gust** | Szczegóły dotyczące wiatru (często powodują opóźnienia). |
| **pressure, visib** | Ciśnienie i widoczność. |
| **time_hour** | Sformatowana data z godziną. |

---

## 2. Architektura i Relacje (Łączenie Danych)

Cała baza `nycflights13` opiera się na relacjach typu **Jeden-do-Wielu (1:N)**. Tabele wymiarów (te mniejsze) przechowują unikalne rekordy ("Jeden"), które w głównej tabeli `flights` mogą występować tysiące razy ("Wiele").

### A. Loty ↔ Linie Lotnicze (`flights` ↔ `airlines`)
* **Relacja:** 1:W (Jedna linia lotnicza, np. Delta, obsługuje tysiące lotów).
* **Klucz Główny (PK - Primary Key):** Kolumna `carrier` w tabeli `airlines`.
* **Klucz Obcy (FK - Foreign Key):** Kolumna `carrier` w tabeli `flights`.

### B. Loty ↔ Samoloty (`flights` ↔ `planes`)
* **Relacja:** 1:W (Jeden konkretny fizyczny samolot wykonuje w roku wiele kursów).
* **Klucz Główny (PK):** Kolumna `tailnum` w tabeli `planes`.
* **Klucz Obcy (FK):** Kolumna `tailnum` w tabeli `flights`.

### C. Loty ↔ Lotniska (`flights` ↔ `airports`)
* **Relacja:** 1:W (Z jednego lotniska wylatuje lub przylatuje na nie wiele samolotów).
* **Klucz Główny (PK):** Kolumna `faa` w tabeli `airports`.
* **Klucz Obcy (FK):** Tu uwaga! Tabele można łączyć na dwa sposoby – używając kolumny `origin` (lotnisko startowe) **LUB** kolumny `dest` (lotnisko docelowe) z tabeli `flights`.

### D. Loty ↔ Pogoda (`flights` ↔ `weather`)
* **Relacja:** 1:W (Jeden historyczny zapis pogody – z konkretnej godziny na konkretnym lotnisku – dotyczy wielu samolotów startujących w tym samym czasie).
* **Klucz Główny Złożony (Composite PK):** Aby jednoznacznie namierzyć pomiar pogody, potrzeba aż 5 kolumn naraz: `origin`, `year`, `month`, `day`, `hour` z tabeli `weather` (albo użyć kombinacji `origin` + `time_hour`).
* **Klucz Obcy Złożony (Composite FK):** Te same 5 kolumn z tabeli `flights` musi zostać spasowanych podczas łączenia.